In [73]:
!pip list

Package                   Version
------------------------- -----------
anyio                     4.15.1
argon2-cffi               25.1.0
argon2-cffi-bindings      26.1.0
arrow                     1.4.0
asttokens                 3.0.2
async-lru                 2.3.0
attrs                     26.1.0
babel                     2.18.0
beautifulsoup4            4.15.0
bleach                    6.4.0
certifi                   2026.7.22
cffi                      2.1.1
charset-normalizer        3.5.1
cloudpickle               3.1.2
colorama                  0.4.6
comm                      0.2.3
debugpy                   1.8.21
defusedxml                0.7.1
executing                 2.2.1
fastjsonschema            2.22.2
fqdn                      1.5.1
h11                       0.16.0
httpcore                  1.0.9
httpx                     0.28.1
idna                      3.19
ipykernel                 7.3.0
ipython                   9.17.1
ipython_pygments_lexers   1.1.1
ipywidgets        


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [74]:
import pandas as pd


In [75]:
df = pd.read_csv("../data/processed/data_wo_cols_20260912_212149.csv")
df.head()

,Unnamed: 0,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure
0,0,1,M14860,M,298.1,308.6,1551,42.8,0,0
1,1,2,L47181,L,298.2,308.7,1408,46.3,3,0
2,2,3,L47182,L,298.1,308.5,1498,49.4,5,0
3,3,4,L47183,L,298.2,308.6,1433,39.5,7,0
4,4,5,L47184,L,298.2,308.7,1408,40.0,9,0


In [76]:
df["Air temperature [K]"].describe()

count    10000.000000
mean       300.004930
std          2.000259
min        295.300000
25%        298.300000
50%        300.100000
75%        301.500000
max        304.500000
Name: Air temperature [K], dtype: float64

In [77]:
df[df["Machine failure"]==1].count()

Unnamed: 0                 339
UDI                        339
Product ID                 339
Type                       339
Air temperature [K]        339
Process temperature [K]    339
Rotational speed [rpm]     339
Torque [Nm]                339
Tool wear [min]            339
Machine failure            339
dtype: int64

In [78]:
df_fallas= df[df["Machine failure"]==1].copy()
print(len(df_fallas),"fallas totales")

339 fallas totales


In [79]:

from sklearn.preprocessing import StandardScaler


df_fallas['temp_diff'] = df['Process temperature [K]'] - df['Air temperature [K]']
df_fallas['power'] = df['Torque [Nm]'] * df['Rotational speed [rpm]'] * 2 * 3.1416 / 60
df_fallas['torque_wear'] = df['Torque [Nm]'] * df['Tool wear [min]']

features = ['Air temperature [K]','Process temperature [K]',
            'Rotational speed [rpm]','Torque [Nm]','Tool wear [min]',
            'temp_diff','power','torque_wear']
X = StandardScaler().fit_transform(df_fallas[features])
X

array([[-0.96036366, -0.87411999,  3.54994987, ...,  0.68474005,
        -3.19445281, -1.54457474],
       [-0.96036366, -0.94755907, -0.2250059 , ...,  0.59873528,
         1.30821091,  1.26798187],
       [-1.00870986, -1.02099815, -0.10793284, ...,  0.59873528,
        -0.53562502,  0.33172449],
       ...,
       [-1.15374844, -0.65380276, -0.35508708, ...,  1.28677345,
         0.75226425,  0.99351226],
       [-1.25044083, -0.72724184, -0.41492442, ...,  1.37277822,
         0.30932552,  1.03332029],
       [-1.10540225, -1.53507169, -0.35248546, ...,  0.16871142,
         1.31858994,  1.07442923]], shape=(339, 8))

In [80]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=5, random_state=42)
clusters = kmeans.fit_predict(X)

df_fallas['cluster'] = clusters
print(df_fallas['cluster'].value_counts())

cluster
3    114
0     89
2     69
1     36
4     31
Name: count, dtype: int64


In [82]:
print(df_raw2.columns.tolist())

['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'F1', 'F2', 'F3', 'F4', 'F5']


In [83]:

cols = ['UDI','Product ID','Type',
        'Air temperature [K]','Process temperature [K]',
        'Rotational speed [rpm]','Torque [Nm]','Tool wear [min]',
        'Machine failure','F1','F2','F3','F4','F5']

df_raw2 = pd.read_csv('../data/raw/ai4i2020.csv', names=cols, header=0)
df_fallas2= df_raw2[df_raw2["Machine failure"]==1].copy()

ultimas_5 = df_fallas2.iloc[:, -5:]

totales = ultimas_5.sum(numeric_only=True)
print(ultimas_5)


      F1  F2  F3  F4  F5
50     0   0   1   0   0
69     0   0   1   1   0
77     1   0   0   0   0
160    0   0   0   1   0
161    0   0   0   1   0
...   ..  ..  ..  ..  ..
9758   1   0   0   0   0
9764   0   0   1   0   0
9822   0   0   0   1   0
9830   0   0   0   1   0
9974   0   0   1   1   0

[339 rows x 5 columns]


In [84]:
kmeans = KMeans(n_clusters=5, random_state=42)
clusters = kmeans.fit_predict(X)

df_fallas['cluster'] = clusters
print(df_fallas['cluster'].value_counts())

cluster
3    114
0     89
2     69
1     36
4     31
Name: count, dtype: int64


In [ ]:
def tipo_real(row):
    for t in ['F1', 'F2', 'F3', 'F4', 'F5']:
        if row[t] == 1:
            return t
    return 'ninguna'

ultimas_5['tipo_real'] = ultimas_5.apply(tipo_real, axis=1)

tabla = pd.crosstab(df_fallas['cluster'], ultimas_5['tipo_real'])
print(tabla)

tipo_real  F1  F2  F3  F4  ninguna
cluster                           
0           0  77   7   0        5
1           5   0  31   0        0
2          15   0   9  42        3
3          26  38  14  36        0
4           0   0  30   0        1
